# 🚀 Qwen2.5-3B + DeepSeek-R1 纯血中文深度思考微调

* **目标**：在 Google Colab **Tesla T4 (15GB)** 显卡上，使用 **Unsloth 4-bit** 快速训练属于你的中文思考模型。
* **耗时预估**：约 **18 ~ 22 分钟**（完全适配剩余使用额度，极度安全）。
* **数据集**：精选 1,000 条纯正中文 DeepSeek-R1 `<think>` 思维链样本，**100% 触发思考机制**，杜绝偷懒与语言漂移。

### 步骤 1：安装高效微调环境（预计 1 分钟）

In [1]:
# 安装 Unsloth 极速微调框架
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 40.7 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.6 MB/s eta 0:00:00:00:0100:01
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for xformers
  Running setup.py clean for xformers
Failed to build xformers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (xformers)
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-ikbbu5kn/unsloth_674926174a01461a8486f8e1ded14fb2
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-ikbbu5kn/unsloth_674926174

### 步骤 2：加载 Qwen2.5-3B 预量化底座并注入 LoRA（预计 1 分钟）

In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048   # 上下文长度，完全适配 T4 显存
load_in_4bit = True     # 4bit 极速量化，显存仅占 ~3.5GB

print("正在加载 Qwen2.5-3B-Instruct 4-bit 底座...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = load_in_4bit,
)

# 配置 LoRA 适配器
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("Qwen2.5-3B LoRA 结构注入成功！")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
正在加载 Qwen2.5-3B-Instruct 4-bit 底座...
==((====))==  Unsloth 2026.10.4: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Unsloth 2026.10.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Qwen2.5-3B LoRA 结构注入成功！


### 步骤 3：下载并清洗纯血中文 DeepSeek-R1 思考数据集（预计 2 分钟）
这里严格过滤：**只保留带有完整 `<think>...</think>` 闭环的样本**，确保模型形成稳定的深思反射！

In [4]:
from datasets import load_dataset

print("正在从 HuggingFace 载入中文 DeepSeek-R1 蒸馏数据集...")
raw_ds = load_dataset("Congliu/Chinese-DeepSeek-R1-Distill-data-110k-SFT", split="train[:2500]")

# 核心清洗逻辑：
# 1. 必须包含 <think> 和 </think>
# 2. 长度过滤：限制在 100 ~ 1800 字符内，杜绝超长样本爆显存或拉长训练时间
def is_valid_cot(sample):
    out = sample.get("output", "")
    return "<think>" in out and "</think>" in out and 100 <= len(out) <= 1800

filtered_ds = raw_ds.filter(is_valid_cot)
target_ds = filtered_ds.select(range(min(1000, len(filtered_ds))))

# 转换为 Qwen 标准 ChatML 格式
SYSTEM_PROMPT = "你是一个具备深度逻辑推演能力的助手。在给出最终结论前，请先在 <think> 与 </think> 标签内进行严密的逐步推导与自我审视。"

def format_chatml(batch):
    texts = []
    for inst, inp, out in zip(batch["instruction"], batch["input"], batch["output"]):
        user_query = f"{inst}\n{inp}".strip() if inp else inst
        text = f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{user_query}<|im_end|>\n<|im_start|>assistant\n{out}<|im_end|>"
        texts.append(text)
    return {"text": texts}

train_dataset = target_ds.map(format_chatml, batched=True, remove_columns=target_ds.column_names)
print(f"数据集准备就绪！共有效样本: {len(train_dataset)} 条（100% 纯正中文思维链）")
print("\n【样本片段预览】:")
print(train_dataset["text"][0][:280] + "...")

正在从 HuggingFace 载入中文 DeepSeek-R1 蒸馏数据集...
数据集准备就绪！共有效样本: 1000 条（100% 纯正中文思维链）

【样本片段预览】:
<|im_start|>system
你是一个具备深度逻辑推演能力的助手。在给出最终结论前，请先在 <think> 与 </think> 标签内进行严密的逐步推导与自我审视。<|im_end|>
<|im_start|>user
大瓶芒果汁容量为30盎司，每瓶售价2700比塞塔。小瓶容量为6盎司，每瓶售价600比塞塔。购买一大瓶而不是同样体积的小瓶可以节省多少比塞塔？<|im_end|>
<|im_start|>assistant
<think>
嗯，我现在要解决这个问题，就是买大瓶芒果汁和同样容量的小瓶相比能省多少钱。题目是说，大瓶有30盎司，每瓶27...


### 步骤 4：启动极速训练（预计 15 ~ 20 分钟）

In [5]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,   # 等效 batch_size = 8
        warmup_steps = 10,
        num_train_epochs = 1,              # 1 个 Epoch 即可形成稳定反射
        learning_rate = 2e-4,
        fp16 = True,
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "qwen_outputs",
    ),
)

print("🚀 开始训练...")
trainer_stats = trainer.train()
print("🎉 训练顺利完成！")

Unsloth: transformers renamed `push_to_hub_token` to `hub_token`. Forwarding your value to `hub_token` - update your code when convenient. If you also passed `hub_token` as None, that is its default here and cannot be distinguished from leaving it unset, so `push_to_hub_token` was used; drop `push_to_hub_token` to keep it.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


🚀 开始训练...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 1 | Total steps = 125
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,2.076777
20,1.656436
30,1.457747
40,1.481627
50,1.529664
60,1.298483
70,1.351580
80,1.397617
90,1.370186
100,1.370791


Unsloth: Restored added_tokens_decoder metadata in qwen_outputs/checkpoint-125/tokenizer_config.json.


🎉 训练顺利完成！


### 步骤 5：现场开炉验丹（经典逻辑题测试）

In [ ]:
FastLanguageModel.for_inference(model)

test_question = "5 只猫用 5 分钟抓了 5 只老鼠。照这个速度，100 只猫抓 100 只老鼠需要多少分钟？"

prompt = f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{test_question}<|im_end|>\n<|im_start|>assistant\n<think>\n"
inputs = tokenizer(prompt, return_tensors = "pt").to("cuda")

print("Qwen2.5 思考中...")
outputs = model.generate(
    **inputs,
    max_new_tokens = 1024,
    temperature = 0.6,
    top_p = 0.95,
)
reply = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=False)
print("<think>\n" + reply)

Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Qwen2.5 思考中...
<think>
嗯，用户问的是9.11和9.9哪个更大。我得仔细想想这个问题。首先，可能需要比较这两个小数的大小。通常，比较小数时，可以逐位比较它们的整数部分和小数部分。

让我回忆一下比较小数的方法。比如，如果比较两个小数的话，比如0.5和0.7，我会先看整数部分，都是0，所以继续看小数点后的第一位。0.5的小数部分是5，0.7的小数部分是7，显然7比5大，所以0.7大于0.5。那同样的方法应该适用于这个情况。

题目中的两个小数是9.11和9.9。首先比较整数部分，都是9，说明两个小数的整数部分相同。接下来比较小数部分。小数部分分别是11和9。这时候，比较11和9，显然11比9大，所以9.11大于9.9。

不过，我得再确认一下自己的步骤是否正确。比如，有没有可能我在比较的时候出错呢？比如，有没有可能小数点后面的位数不同导致比较结果不同？例如，如果其中一个小数有三位小数，另一个只有两位，这时候可能需要补零来比较。但是在这个例子中，两个小数都是两位小数，所以不需要补零。

另外，用户可能想知道为什么这样比较。比如，如果比较0.8和0.12，会把0.8写作0.80，然后比较0.80和0.12。这其实是在处理小数点后位数不同的情况，但在这里两个小数都有两位小数，所以不需要处理。所以这样的方法应该是正确的。

</think>



要比较9.11和9.9的大小，可以通过以下步骤进行逐位比较：

1. **整数部分**：两者的整数部分均为9，相等。
   
2. **小数部分**：
   - 9.11的小数部分为11，对应于十进制中的1.1。
   - 9.9的小数部分为9，对应于十进制中的0.9。

3. **比较小数部分**：
   - 比较11（即0.11）和9（即0.9）。显然，11 > 9。

因此，根据整数部分相同且小数部分9 > 11，可知9.9 > 9.11。

\boxed{9.9} 是更大的数。<|im_end|>


### 步骤 6：保存权重并一键打包下载到本地

In [7]:
import os

save_dir = "my_qwen_r1_lora"
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"权重已保存至目录: {save_dir}")

# 自动压缩为 zip 文件
zip_file = f"{save_dir}.zip"
os.system(f"zip -r {zip_file} {save_dir}")
print(f"📦 已打包生成压缩包: {zip_file}（约 80MB~120MB）")

# 触发浏览器自动下载
try:
    from google.colab import files
    print("正在拉起浏览器下载...")
    files.download(zip_file)
except Exception as e:
    print("请在左侧文件栏右键点击 my_qwen_r1_lora.zip 自行下载。")

Unsloth: Restored added_tokens_decoder metadata in my_qwen_r1_lora/tokenizer_config.json.


权重已保存至目录: my_qwen_r1_lora
📦 已打包生成压缩包: my_qwen_r1_lora.zip（约 80MB~120MB）
正在拉起浏览器下载...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>